# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and record data from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Accessing the metadata as an object (not as dict/list)
metadata = dataset.metadata
print("Dataset Title: ", metadata.name)
print("Description: ", metadata.description)
print("Published: ", metadata.datePublished)
print("Keywords: ", metadata.keywords)
print("License: ", metadata.license)
print("Dataset ID (@id):", metadata.id)
print("Dataset version: ", metadata.version)

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

All entities, including record sets and fields, are referenced by their `@id`. This ensures consistent referencing throughout the notebook.

In [ ]:
# List all record sets and their fields, referencing by @id
record_sets = dataset.record_sets
print("Available Record Sets:")
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id}, name: {rs.name}")
    if rs.fields is not None:
        print("  Fields:")
        for f in rs.fields:
            print(f"    - Field @id: {f.id}, name: {f.name}, data type: {f.data_type}")
    print()
# Preview sample records for each record set
for rs in record_sets:
    print(f"Sample records from RecordSet '{rs.name}' (@id: {rs.id}):")
    for i, record in enumerate(dataset.records(record_set=rs.id)):
        pprint.pprint(record)
        if i >= 2:
            break  # Show only 3 records per set
    print("---\n")

## 3. Data Extraction
Load data from each record set into a DataFrame, referencing each set and field via their `@id`.

You can analyze specific columns by using the respective field's `@id` shown in the overview.

In [ ]:
# Extract data from each record set
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Columns for RecordSet @id='{rs_id}': {df.columns.tolist()}")
    print("Preview:")
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Perform data processing steps such as filtering, normalizing, and grouping using field `@id`s.

- Remove outliers
- Normalize numeric fields
- Group by key attributes

Below, select a numeric field and a grouping field for analysis using their `@id`.

In [ ]:
# Choose a record set and field @id for EDA
# For illustration, let's use the first available record set and its fields
first_rs_id = record_set_ids[0]
df = dataframes[first_rs_id]

# Identify numeric field and group field by their @id (replace with actual IDs if needed)
numeric_fields = [f for f in dataset.record_sets[0].fields if f.data_type in ['Integer', 'Float', 'Number']]
group_fields = [f for f in dataset.record_sets[0].fields if f.data_type == 'Text' or f.data_type == 'String']
if len(numeric_fields) == 0:
    print("No numeric fields found.")
else:
    numeric_field_id = numeric_fields[0].id  # Use the first numeric field's @id

    print(f"Numeric field selected: {numeric_field_id}")

    # Set a threshold for filtering
    threshold = df[numeric_field_id].mean() if numeric_field_id in df else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Grouping by a field
    if len(group_fields) > 0:
        group_field_id = group_fields[0].id
        if group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
            display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset, referencing fields by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of the selected numeric field
if len(numeric_fields) > 0 and numeric_field_id in df:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=12, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if len(group_fields) > 0 and group_field_id in df:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to load, explore, and process the FAIR² colorectal cancer dataset using the `mlcroissant` library.

- **Metadata inspection**: Understand dataset content and provenance.
- **Record sets and fields**: All referenced consistently by their `@id`.
- **Data extraction and EDA**: Filtering, normalization, and grouping using field and record set `@id`s.
- **Visualization**: Distributions and relationships between key clinical fields.

These steps support advanced biomedical research, and additional analysis can be performed by referencing specific record sets and fields by their unique `@id` in the dataset schema.